# Data Agent (Morgane)

**What this notebook does**
1. **Download** – daily prices of our 50 ETFs from Yahoo Finance (raw close + adjusted close + volume + currency).
2. **Clean** – fixes the Yahoo errors we checked by hand (unadjusted splits, corrupted day) and adds the **leverage** factor of each ETF.
3. **Check** – flags data problems (gaps, extreme moves, zero-volume prints, stale prices, non-EUR prices).
4. **Metrics** – returns, volatility and drawdowns per ETF, for the Risk agent and the Portfolio Manager.
5. **LangGraph** – a small agent that decides whether to download or update, runs the checks and writes a short data report for the team.

**What the others read** (do not rename these files or columns without telling the team)

| File | Columns used | Used by |
|---|---|---|
| `data/prices.csv` | `date`, `code`, `theme`, `close`, `adj_close`, `leverage` | Gonçalo (Macro/Risk), Francesco (Portfolio Manager), Technical agent |
| `data/metrics.csv` | one row per ETF: returns, volatility, drawdowns, leverage, liquidity | Portfolio Manager (AI version), Risk |
| `data/data_issues.csv` | one row per detected problem | everyone (to know which data to trust) |
| `data/data_report.md` | the AI's short data report | Portfolio Manager memo, final report |

**How to read it**
- Lines ending with `# <<< ADAPT` are the parts to change.
- Parts 1–5 work **without** the AI. The AI data report is **off** by default (`USE_AI_DATA_REPORT = False`, to save the free Gemini quota); to switch it on, set it to `True` and add the same Gemini key as the other notebooks (`LLM_API_KEY` in `.env` or Colab Secrets).
- **Francesco's `Portfolio_Manager/team_graph.ipynb` contains a copy of parts 3–8.** Change the code **here first**, then copy it there.
- Run the cells **from top to bottom**.

## 0. Install (once)

In [ ]:
%pip install -q pandas numpy yfinance requests langgraph langchain-openai openai python-dotenv

## 1. ✏️ Settings

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd

# Find the project folder (the one with data/tickers.csv), same logic as the other notebooks
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "data" / "tickers.csv").exists():
        os.chdir(folder)
        break
print("Project folder:", Path.cwd())

DATA_DIR     = Path("data")
TICKERS_FILE = DATA_DIR / "tickers.csv"      # our 50 ETFs + Yahoo symbols (8 fixed by hand: keep it on GitHub)
PRICES_FILE  = DATA_DIR / "prices.csv"       # main output, read by everyone
METRICS_FILE = DATA_DIR / "metrics.csv"
ISSUES_FILE  = DATA_DIR / "data_issues.csv"
REPORT_FILE  = DATA_DIR / "data_report.md"

START_DATE = "2021-01-01"                    # <<< ADAPT  about 5 years of history

# Leverage factor of each leveraged / inverse ETF (from the official product names). All other ETFs = 1.
LEVERAGE = {                                 # <<< ADAPT  check each one on the issuer's factsheet
    "NASDAQ 3X": 3,       # WisdomTree NASDAQ 100 3x Daily Leveraged
    "COPPER LVG": 3,      # WisdomTree Copper 3x Daily Leveraged
    "PALLADIUM LVG": 2,   # WisdomTree Palladium 2x Daily Leveraged
    "EURO STOXX 2X": 2,   # Amundi EURO STOXX 50 Daily (2x) Leveraged
    "S&P 500 SHORT": -1,  # Xtrackers S&P 500 Inverse Daily Swap
    "STOXX SHORT": -2,    # Amundi EURO STOXX 50 Daily (-2x) Inverse
}

# ---- Data fixes: known Yahoo errors, each one checked by hand ----
# Splits Yahoo did not adjust: {code: (first date at the NEW price, ratio)} -> earlier prices divided by ratio
SPLITS = {                                   # <<< ADAPT  add a line if 'check' finds a new split
    "EURO BANKS":    ("2026-10-26", 80),     # 80:1 split (349.80 -> 4.43)
    "PALLADIUM LVG": ("2022-12-19", 4),      # 4:1 split, WisdomTree announcement 19 Dec 2022
}
# Days where Yahoo's data is corrupted -> removed for all ETFs
BAD_DATES = [                                # <<< ADAPT
    "2025-10-24",   # 39/50 ETFs missing, the rest mostly zero volume with fake +13-21% jumps
]
# Genuine extreme moves we checked and keep on purpose (no fix):
#   COPPER LVG 2025-07-31 -64.5%: COMEX copper -22% after the US tariff decision, x3 leverage

# ---- Check thresholds (documented assumptions, see report) ----
TRADING_DAYS     = 252     # trading days per year (annualised volatility)
FFILL_LIMIT      = 5       # carry the last price forward for at most 5 trading days
LONG_GAP_DAYS    = 5       # a gap longer than a trading week = data problem, not a holiday
EXTREME_MOVE     = 0.20    # daily move above 20% -> manual review
SUSPICIOUS_MOVE  = 0.08    # move above 8% on zero volume that reverses next day -> likely bad price
SHORT_HISTORY    = 252     # less than ~1 year of data -> long-term metrics not reliable
EXPECTED_CURRENCY = "EUR"
USE_AI_DATA_REPORT = False   # <<< ADAPT  True = AI data report (several extra Gemini calls: free quota!)
WINDOWS = {"1w": 5, "1m": 21, "3m": 63, "6m": 126, "1y": 252}   # in trading days

YAHOO_SUFFIX  = {"Xetra": ".DE", "Brussels": ".BR", "Milan": ".MI"}
FIGI_EXCHANGE = {"Xetra": "GY", "Brussels": "BB", "Milan": "IM"}   # OpenFIGI (Bloomberg) exchange codes

## 2. ETF list and Yahoo symbols

`data/tickers.csv` holds our 50 ETFs (`isin`, `code`, `name`, `theme`, `exchange`, `yahoo`). It is **never overwritten**:
8 symbols were fixed by hand (e.g. NASDAQ 3X uses the Milan listing `QQQ3.MI`, Xetra is not on Yahoo).
`fill_missing_tickers()` only fills **empty** `yahoo` cells, from the ISIN + exchange (OpenFIGI, then Yahoo search).

In [ ]:
import requests
import yfinance as yf

tickers = pd.read_csv(TICKERS_FILE).fillna("")
print(f"{len(tickers)} ETFs, {int((tickers['yahoo'] == '').sum())} without a Yahoo symbol")


def fill_missing_tickers():
    """Fill only the empty 'yahoo' cells of tickers.csv (never changes a symbol already there)."""
    t = pd.read_csv(TICKERS_FILE).fillna("")
    todo = t[t["yahoo"] == ""]
    if todo.empty:
        print("All symbols already filled."); return t
    jobs = [{"idType": "ID_ISIN", "idValue": r.isin, "exchCode": FIGI_EXCHANGE[r.exchange]} for r in todo.itertuples()]
    try:
        answers = requests.post("https://api.openfigi.com/v3/mapping", json=jobs[:10], timeout=30).json()
    except Exception as e:
        answers = []; print("OpenFIGI error:", e)
    for (i, r), a in zip(todo.iterrows(), answers):
        data = a.get("data") or []
        if data and data[0].get("ticker"):
            t.loc[i, "yahoo"] = data[0]["ticker"] + YAHOO_SUFFIX[r.exchange]
    for i, r in t[t["yahoo"] == ""].iterrows():          # fallback: Yahoo search by ISIN
        try:
            found = [q["symbol"] for q in yf.Search(r.isin, max_results=20).quotes
                     if q.get("symbol", "").endswith(YAHOO_SUFFIX[r.exchange])]
            t.loc[i, "yahoo"] = found[0] if found else ""
        except Exception:
            pass
    t.to_csv(TICKERS_FILE, index=False)
    print("Still missing (find by hand on justETF):", ", ".join(t.loc[t["yahoo"] == "", "code"]) or "none")
    return t

## 3. Download + clean → `data/prices.csv`

- `full()`: whole history since `START_DATE` (safest: re-run it before each trading day).
- `update()`: only the missing days (needs a file made by `full()`).
- Fixes are applied **only to newly downloaded raw rows**, so a split is never divided twice.
- **Today's row is always dropped** (Paris time): before the close Yahoo gives intraday prices, and orders are executed at the previous day's close (T-1).
- Columns written: `date, code, isin, yahoo, theme, currency, close, adj_close, volume, leverage, return`.
  `close` = raw price (orders are executed at the previous day's close), `adj_close` = with dividends (performance).

In [ ]:
COLUMNS = ["date", "code", "isin", "yahoo", "theme", "currency", "close", "adj_close", "volume", "leverage", "return"]


def today_paris():
    """Today's date in Brussels/Paris time (Colab runs in UTC)."""
    return pd.Timestamp.now(tz="Europe/Paris").normalize().tz_localize(None)


def get_currencies(symbols) -> dict:
    """Trading currency of each Yahoo symbol (should be EUR for all our listings)."""
    out = {}
    for s in symbols:
        try:
            out[s] = yf.Ticker(s).fast_info["currency"]
        except Exception:
            out[s] = "UNKNOWN"
    return out


def download(start: str, currencies: dict | None = None) -> pd.DataFrame:
    """Raw Yahoo data, one row per (date, ETF). One single request for all ETFs (Yahoo blocks too many)."""
    t = pd.read_csv(TICKERS_FILE).fillna("")
    t = t[t["yahoo"] != ""]
    raw = yf.download(t["yahoo"].tolist(), start=start, auto_adjust=False,   # raw close AND adjusted close
                      group_by="ticker", progress=False, threads=True, timeout=30)
    if raw is None or raw.empty:
        raise RuntimeError("Yahoo returned no data: check the connection or try again in a few minutes")
    currencies = currencies or get_currencies(t["yahoo"])
    rows = []
    for r in t.itertuples():
        if r.yahoo not in raw.columns.get_level_values(0):
            print(f"WARNING: no data for {r.code} ({r.yahoo})"); continue
        df = raw[r.yahoo][["Close", "Adj Close", "Volume"]].dropna(subset=["Close"]).reset_index()
        df.columns = ["date", "close", "adj_close", "volume"]
        df = df.assign(code=r.code, isin=r.isin, yahoo=r.yahoo, theme=r.theme, currency=currencies.get(r.yahoo, "UNKNOWN"))
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


def clean(data: pd.DataFrame) -> pd.DataFrame:
    """Fix known Yahoo errors + add leverage. Only for RAW downloaded rows (never twice on the same rows)."""
    data = data.copy()
    data["date"] = pd.to_datetime(data["date"])
    data = data[~data["date"].isin(pd.to_datetime(BAD_DATES))]                 # 1. corrupted days
    data = data[data["date"] < today_paris()]       # (Francesco) today's row is intraday, not a close: orders use T-1
    for c, (split_date, ratio) in SPLITS.items():                               # 2. unadjusted splits
        mask = (data["code"] == c) & (data["date"] < pd.Timestamp(split_date))
        data.loc[mask, ["close", "adj_close"]] /= ratio
    data["leverage"] = data["code"].map(LEVERAGE).fillna(1).astype(int)        # 3. leverage factor
    return data


def add_returns(data: pd.DataFrame) -> pd.DataFrame:
    data = data.sort_values(["code", "date"]).reset_index(drop=True)
    data["return"] = data.groupby("code")["adj_close"].pct_change(fill_method=None)   # dividends included
    return data


def full():
    data = add_returns(clean(download(START_DATE)))
    data[COLUMNS].to_csv(PRICES_FILE, index=False)
    print(f"Done: {data['code'].nunique()} ETFs, {data['date'].min():%Y-%m-%d} to {data['date'].max():%Y-%m-%d}")


def update():
    old = pd.read_csv(PRICES_FILE, parse_dates=["date"])
    if "leverage" not in old.columns:
        raise RuntimeError("Old price file without fixes: run full() once")
    start = (old["date"].max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")   # only NEW days
    currencies = old.drop_duplicates("yahoo").set_index("yahoo")["currency"].to_dict()
    try:
        new = clean(download(start, currencies))
    except RuntimeError:                 # Yahoo has no new day yet (weekend, holiday, before the close)
        print("No new prices on Yahoo yet. Last date:", f"{old['date'].max():%Y-%m-%d}"); return
    new = new[new["date"] > old["date"].max()]
    if new.empty:
        print("Already up to date:", f"{old['date'].max():%Y-%m-%d}"); return
    data = add_returns(pd.concat([old.drop(columns="return"), new]).drop_duplicates(["date", "code"], keep="last"))
    data[COLUMNS].to_csv(PRICES_FILE, index=False)
    print(f"Done: up to {data['date'].max():%Y-%m-%d} (+{new['date'].nunique()} days)")

# full()     # <<< ADAPT  uncomment to download now (part 6: the agent decides by itself)

## 4. Quality checks → `data/data_issues.csv`

Nothing is changed here: issues are listed for **review**. If a new one is a split or a corrupted day,
add it to `SPLITS` / `BAD_DATES` (part 1) and re-run `full()`.

In [ ]:
def load_prices() -> pd.DataFrame:
    df = pd.read_csv(PRICES_FILE, parse_dates=["date"])
    df = df[df["date"] < today_paris()]  # (Francesco) never use today's intraday prices
    if "leverage" not in df.columns:     # old raw file from before the fixes: clean it in memory
        print("Raw prices.csv (no fixes yet): cleaning in memory. Run full() to save a clean file.")
        df = add_returns(clean(df.drop(columns=["return"], errors="ignore")))
        df["currency"] = df.get("currency", "UNKNOWN")
    return df.sort_values(["code", "date"]).reset_index(drop=True)


def detect_issues(df: pd.DataFrame) -> pd.DataFrame:
    issues = []
    add = lambda c, d, kind, detail: issues.append({"code": c, "date": d, "issue": kind, "detail": detail})
    start, end = df["date"].min(), df["date"].max()
    for c, g in df.groupby("code"):
        g = g.set_index("date").sort_index()
        lev, cur = int(g["leverage"].iloc[0]), g["currency"].iloc[0]
        ret = g["adj_close"].pct_change(fill_method=None)
        if cur != EXPECTED_CURRENCY:
            add(c, None, "currency_not_eur", f"currency '{cur}'")
        if (g["close"] <= 0).any():
            add(c, None, "non_positive_price", f"{int((g['close'] <= 0).sum())} rows <= 0")
        if len(g) < SHORT_HISTORY:
            add(c, g.index[0].date(), "short_history", f"only {len(g)} trading days")
        elif g.index[0] > start + pd.Timedelta(days=30):
            add(c, g.index[0].date(), "late_start", "starts later than the other ETFs")
        if g.index[-1] < end:
            add(c, g.index[-1].date(), "stale", f"no price after {g.index[-1]:%Y-%m-%d}")
        missing = ~pd.bdate_range(g.index[0], g.index[-1]).isin(g.index)
        runs = pd.Series(missing).groupby((~pd.Series(missing)).cumsum()).sum()
        if len(runs) and runs.max() > LONG_GAP_DAYS:
            add(c, None, "long_gap", f"longest gap {int(runs.max())} trading days")
        for d, x in ret[ret.abs() > EXTREME_MOVE].items():
            note = f" (leverage {lev:+d}x: may be genuine)" if lev != 1 else ""
            add(c, d.date(), "extreme_move", f"{x:+.1%} in one day{note}")
        nxt = ret.shift(-1)
        bad = (ret.abs() > SUSPICIOUS_MOVE) & (g["volume"] == 0) & (np.sign(nxt) == -np.sign(ret)) & (nxt.abs() > 0.5 * ret.abs())
        for d in ret[bad].index:
            add(c, d.date(), "suspicious_print", f"{ret[d]:+.1%} on zero volume, then {nxt[d]:+.1%}")
    return pd.DataFrame(issues, columns=["code", "date", "issue", "detail"])

## 5. Metrics → `data/metrics.csv`

One row per ETF, computed on the **adjusted** close (dividends count as performance). Gaps are carried forward
for at most 5 days, never filled before an ETF's launch and never interpolated (that would use future prices).

In [ ]:
def max_drawdown(p: pd.Series) -> float:
    p = p.dropna()
    return float((p / p.cummax() - 1).min()) if len(p) else np.nan


def compute_metrics(df: pd.DataFrame) -> pd.DataFrame:
    adj = df.pivot(index="date", columns="code", values="adj_close").sort_index().ffill(limit=FFILL_LIMIT)
    rets = adj.pct_change(fill_method=None)
    info = df.drop_duplicates("code").set_index("code")
    rows = []
    for c in adj.columns:
        p, r = adj[c].dropna(), rets[c].dropna()
        vol = df.loc[df["code"] == c].set_index("date")["volume"]
        row = {"code": c, "theme": info.loc[c, "theme"], "leverage": int(info.loc[c, "leverage"]),
               "currency": info.loc[c, "currency"], "first_date": p.index[0].date(), "last_date": p.index[-1].date(),
               "history_days": len(p), "last_close": float(df.loc[df["code"] == c, "close"].iloc[-1])}
        for name, n in WINDOWS.items():
            row[f"ret_{name}"] = p.iloc[-1] / p.iloc[-n - 1] - 1 if len(p) > n else np.nan
        for name in ["1m", "3m", "1y"]:
            n = WINDOWS[name]
            row[f"vol_{name}"] = r.iloc[-n:].std() * np.sqrt(TRADING_DAYS) if len(r) >= n else np.nan
        row["max_drawdown_all"] = max_drawdown(p)
        row["max_drawdown_1y"] = max_drawdown(p.iloc[-WINDOWS["1y"]:]) if len(p) > WINDOWS["1y"] else np.nan
        row["current_drawdown"] = p.iloc[-1] / p.max() - 1
        row["avg_volume_1m"] = vol.iloc[-WINDOWS["1m"]:].mean()
        row["zero_volume_days_1m"] = int((vol.iloc[-WINDOWS["1m"]:] == 0).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("code")


def build_outputs() -> dict:
    """Checks + metrics, saved for the other agents. Returns a short summary."""
    df = load_prices()
    metrics, issues = compute_metrics(df), detect_issues(df)
    metrics.to_csv(METRICS_FILE)
    issues.to_csv(ISSUES_FILE, index=False)
    return {"n_etfs": int(metrics.shape[0]),
            "date_range": [f"{df['date'].min():%Y-%m-%d}", f"{df['date'].max():%Y-%m-%d}"],
            "issues_by_type": issues["issue"].value_counts().to_dict(),
            "leverage_factors": metrics.loc[metrics["leverage"] != 1, "leverage"].to_dict(),
            "currencies": metrics["currency"].value_counts().to_dict()}


# Quick test without AI
print(json.dumps(build_outputs(), indent=1, default=str))
pd.read_csv(METRICS_FILE, index_col="code")[["theme", "leverage", "ret_1m", "vol_1y", "max_drawdown_all"]].head()

## 6a. Gemini settings

Same setup as Gonçalo's and Francesco's notebooks: key from **Colab Secrets** (`LLM_API_KEY`) or from the **`.env`** file
(never on GitHub). With `USE_AI_DATA_REPORT = False`, Gemini is not called at all (not even to test the model).

In [ ]:
try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
except Exception:
    from dotenv import load_dotenv; load_dotenv(Path(".env"))   # .env in the project folder
os.environ.setdefault("LLM_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")

CANDIDATE_MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-flash-latest"]   # <<< ADAPT
os.environ["LLM_MODEL"] = ""
if USE_AI_DATA_REPORT and os.environ.get("LLM_API_KEY"):     # no Gemini call at all when the AI report is off
    from openai import OpenAI
    client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"], timeout=60, max_retries=2)
    for m in CANDIDATE_MODELS:
        try:
            client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Say hi"}])
            os.environ["LLM_MODEL"] = m; break
        except Exception as e:
            print(f"{m}: not available ({type(e).__name__})")
print("Gemini model:", os.environ["LLM_MODEL"] or "none (AI data report off or no key: parts 1-5 still work)")

## 6b. Tools for the agent

The AI never computes a number: every figure comes from these Python tools (results are short JSON).

In [ ]:
from langchain_core.tools import tool

_j = lambda x: json.dumps(x, default=str, indent=1)
_r = lambda d: {k: (round(v, 4) if isinstance(v, (float, np.floating)) and not np.isnan(v) else v) for k, v in d.items()}


@tool
def get_data_status() -> str:
    """Is there a price file, what dates does it cover, is it up to date? Call it first."""
    last_bday = (today_paris() - pd.offsets.BDay(1)).date()
    if not PRICES_FILE.exists():
        return _j({"price_file_exists": False, "last_business_day": last_bday})
    df = pd.read_csv(PRICES_FILE, parse_dates=["date"])
    return _j({"price_file_exists": True, "has_fixes": "leverage" in df.columns,
               "first_date": df["date"].min().date(), "last_date": df["date"].max().date(),
               "n_etfs": df["code"].nunique(), "last_business_day": last_bday,
               "up_to_date": df["date"].max().date() >= last_bday})


@tool
def refresh_prices(mode: str) -> str:
    """Download prices. mode='full' if there is no file or it has no fixes, mode='update' if it is only out of date."""
    try:
        full() if mode == "full" else update() if mode == "update" else None
    except Exception as e:
        return f"Download failed: {e}. Continue with the existing file."
    return get_data_status.invoke({})


@tool
def build_dataset() -> str:
    """Run the quality checks and compute the metrics; saves metrics.csv and data_issues.csv for the team."""
    return _j(build_outputs())


@tool
def list_data_issues(issue_type: str = "all") -> str:
    """Detected issues. issue_type: all, extreme_move, suspicious_print, long_gap, short_history, late_start,
    stale, non_positive_price, currency_not_eur."""
    i = pd.read_csv(ISSUES_FILE)
    i = i if issue_type == "all" else i[i["issue"] == issue_type]
    return _j(i.to_dict(orient="records")) if len(i) else "No issues of this type."


@tool
def get_risk_overview(sort_by: str = "vol_1y", top_n: int = 8) -> str:
    """Rank ETFs by a metric (vol_1y, max_drawdown_all, ret_1m, ret_3m, current_drawdown...)."""
    m = pd.read_csv(METRICS_FILE, index_col="code")
    if sort_by not in m.columns:
        return f"Unknown metric. Available: {', '.join(m.columns)}"
    top = m.sort_values(sort_by, ascending="drawdown" in sort_by).head(top_n)
    cols = list(dict.fromkeys(["theme", "leverage", sort_by, "vol_1y", "max_drawdown_all"]))
    return _j({c: _r(r) for c, r in top[cols].to_dict(orient="index").items()})


@tool
def get_etf_metrics(code: str) -> str:
    """All metrics and issues of one ETF, by its code (e.g. 'WORLD', 'NASDAQ 3X')."""
    m = pd.read_csv(METRICS_FILE, index_col="code")
    if code not in m.index:
        return f"Unknown code. Available: {', '.join(m.index)}"
    i = pd.read_csv(ISSUES_FILE)
    return _j({"metrics": _r(m.loc[code].to_dict()), "issues": i[i["code"] == code].to_dict(orient="records")})


tools = [get_data_status, refresh_prices, build_dataset, list_data_issues, get_risk_overview, get_etf_metrics]
print(get_data_status.invoke({}))

## 7. ✏️ Mini-agent with LangGraph

Same loop as the demo notebook: the AI calls tools until it has what it needs, then writes the data report.

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

SYSTEM_PROMPT = """You are the Data agent of an ETF portfolio team. You run before the Technical, Macro/News, Risk and
Portfolio Manager agents, who read data/prices.csv and data/metrics.csv.

1. Call get_data_status. No file or has_fixes=false -> refresh_prices('full'). Not up to date -> refresh_prices('update').
   If a download fails, say so and continue with the existing file.
2. Call build_dataset.
3. Review the issues with list_data_issues: for each extreme_move or suspicious_print, say if it looks like a data error
   or a genuine move (leveraged/inverse ETFs can move a lot). Mention any ETF not in EUR.
4. Use get_risk_overview to name the most volatile ETFs and the deepest drawdowns.
5. Write a short data report (max 15 lines): date range, number of ETFs, issues to watch, ETFs with short history,
   leveraged/inverse ETFs and their factor, highest-risk ETFs.

Rules: every number must come from a tool result. Never estimate or invent figures or dates.
You do not recommend trades: that is the Portfolio Manager's job."""   # <<< ADAPT prompt


def get_llm():
    if not all(os.environ.get(k) for k in ("LLM_BASE_URL", "LLM_MODEL", "LLM_API_KEY")):
        return None
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"],
                      model=os.environ["LLM_MODEL"], temperature=0, timeout=90, max_retries=3)


def agent_node(state: MessagesState):
    return {"messages": [get_llm().bind_tools(tools).invoke([SystemMessage(SYSTEM_PROMPT)] + state["messages"])]}


g = StateGraph(MessagesState)
g.add_node("agent", agent_node)
g.add_node("tools", ToolNode(tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
g.add_edge("tools", "agent")
data_agent_app = g.compile()
print(data_agent_app.get_graph().draw_mermaid())

## 8. Run it

In [ ]:
def run_data_agent(use_ai=False) -> str:
    """Refreshes prices + checks + metrics. With use_ai=True, the AI agent also writes a data report."""
    if not use_ai or get_llm() is None:
        status = json.loads(get_data_status.invoke({}))
        if not status["price_file_exists"] or not status["has_fixes"]:
            full()
        elif not status["up_to_date"]:
            update()
        report = "[no AI data report] " + json.dumps(build_outputs(), default=str)
    else:
        out = data_agent_app.invoke({"messages": [{"role": "user", "content": "Prepare today's market data and write the data report."}]},
                                    {"recursion_limit": 30})
        report = out["messages"][-1].content
        for m in out["messages"]:                  # trace of the tools the AI used (keep it for the jury)
            for call in getattr(m, "tool_calls", None) or []:
                print("tool:", call["name"], call["args"])
    REPORT_FILE.write_text(report, encoding="utf-8")
    return report

print(run_data_agent(use_ai=USE_AI_DATA_REPORT))

## 9. Link with the team graph

Francesco's `Portfolio_Manager/team_graph.ipynb` runs the whole flow:
`data → macro → portfolio manager → validation → human approval → orders → risk → log`.
Its **data** node uses a copy of this notebook's code (parts 3–8): it calls `run_data_agent(use_ai=USE_AI_DATA_REPORT)`,
then reloads the prices so every agent uses the same fresh numbers.

**To keep both notebooks identical:** change the code here, then copy these functions into `team_graph.ipynb`:
`today_paris`, `download`, `clean`, `add_returns`, `full`, `update`, `load_prices`, `detect_issues`, `compute_metrics`,
`build_outputs`, the tools, `run_data_agent`, and the settings `LEVERAGE`, `SPLITS`, `BAD_DATES`.

## 10. Still to decide / do

| # | What | With whom |
|---|------|-----------|
| 1 | Confirm the leverage factors on each issuer's factsheet | Me |
| 2 | Run this notebook and commit the clean `data/prices.csv` (the one on `main` has no fixes and no `leverage` column) | Me |
| 3 | Copy the latest version of parts 3–8 into `team_graph.ipynb` after each change here | Francesco + me |
| 4 | Technical agent: reads `prices.csv` (`adj_close`) for moving averages / momentum | Mingzhu + me |
| 5 | Optional: anomaly threshold based on each ETF's own volatility instead of a fixed 20% | Me |